In [1]:
%load_ext ngql

# connect ngql jupyter extension to nebulagraph
%ngql --address 127.0.0.1 --port 9669 --user root --password nebula
# create a new space
%ngql CREATE SPACE IF NOT EXISTS langchain(partition_num=1, replica_factor=1, vid_type=fixed_string(128));

Connection Pool Created


""


In [2]:
# Wait for a few seconds for the space to be created.
%ngql USE langchain;

""


In [3]:
%%ngql
CREATE TAG IF NOT EXISTS movie(name string);
CREATE TAG IF NOT EXISTS person(name string, birthdate string);
CREATE EDGE IF NOT EXISTS acted_in();
CREATE TAG INDEX IF NOT EXISTS person_index ON person(name(128));
CREATE TAG INDEX IF NOT EXISTS movie_index ON movie(name(128));

""


In [4]:
%%ngql
INSERT VERTEX person(name, birthdate) VALUES "Al Pacino":("Al Pacino", "1940-04-25");
INSERT VERTEX movie(name) VALUES "The Godfather II":("The Godfather II");
INSERT VERTEX movie(name) VALUES "The Godfather Coda: The Death of Michael Corleone":("The Godfather Coda: The Death of Michael Corleone");
INSERT EDGE acted_in() VALUES "Al Pacino"->"The Godfather II":();
INSERT EDGE acted_in() VALUES "Al Pacino"->"The Godfather Coda: The Death of Michael Corleone":();

""


In [4]:
from langchain.chains import NebulaGraphQAChain
from langchain_community.graphs import NebulaGraph
# from langchain_openai import ChatOpenAI


In [5]:
graph = NebulaGraph(
    space="langchain",
    username="root",
    password="nebula",
    address="127.0.0.1",
    port=9669,
    session_pool_size=30,
)

In [6]:
print(graph.get_schema)

Node properties: [{'tag': 'movie', 'properties': [('name', 'string')]}, {'tag': 'person', 'properties': [('name', 'string'), ('birthdate', 'string')]}]
Edge properties: [{'edge': 'acted_in', 'properties': []}]
Relationships: ['(:person)-[:acted_in]->(:movie)']



In [ ]:
# Models and its types
# repo_id = "nomic-ai/gpt4all-falcon-ggml", model_name = "gpt4all-falcon-ggml", model_type = "gptj", model_file_name = "models/ggml-model-gpt4all-falcon-q4_0.bin"
# repo_id = "nomic-ai/gpt4all-13b-snoozy", model_name = "gpt4all-13b-snoozy", model_type = "llama", model_file_name =


In [ ]:
from langchain_community.llms import CTransformers

# Working Version
llm = CTransformers(
    model = "TheBloke/Llama-2-13B-chat-GGUF",
    model_type = "llama",
    model_file = "llama-2-13b-chat.Q4_K_S.gguf",
    lib = "avx2",
    max_new_tokens = 1024,
    temperature = 0.5
)

In [5]:
from langchain_community.document_loaders.csv_loader import CSVLoader

loader = CSVLoader(file_path="bbc_news_article.csv")
documents = loader.load()

In [11]:
from llama_index import ServiceContext, KnowledgeGraphIndex
from llama_index.graph_stores import SimpleGraphStore
from llama_index.storage.storage_context import StorageContext
from langchain.embeddings import HuggingFaceBgeEmbeddings

embeddings = HuggingFaceBgeEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2',
                                       model_kwargs={'device': 'cpu'})


HuggingFaceBgeEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={}, query_instruction='Represent this question for searching relevant passages: ')

In [ ]:
# Create Service Context
service_context = ServiceContext.from_defaults(llm=embeddings, chunk_size=512)
graph_store = SimpleGraphStore()
storage_context = StorageContext.from_defaults(graph_store=graph_store)

index = KnowledgeGraphIndex.from_documents(
    documents,
    max_triplets_per_chunk=2,
    storage_context=storage_context,
    service_context=service_context
)

In [ ]:
chain = NebulaGraphQAChain.from_llm(
    llm=llm, graph=graph, verbose=True
)

chain.run("List all entities from the schema?")